# PyOccam Demo Notebook
## Clean Implementation for Search, Fit, and Analysis

This notebook demonstrates the core OCCAM workflow:
1. Load data
2. Search for best models
3. Fit selected models
4. Analyze results
5. Save outputs

In [1]:
# Import required libraries
import pyoccam
import os
import pandas as pd
from datetime import datetime

print(f"PyOccam version: {pyoccam.__version__}")

PyOccam 0.1.2 loaded successfully
PyOccam 0.1.2 loaded. Type pyoccam.help() for usage.
PyOccam version: 0.1.2


## 1. Configuration

Set your analysis parameters here:

In [5]:
# Configuration parameters
DATA_FILE = "fire_data_split_all_signature_groups_nlcd_rebinned.txt"           # Your data file
SEARCH_TYPE = "full-up"            # Options: loopless-up, full-up, disjoint-up, chain-up
SEARCH_LEVELS = 7                      # Search depth (1-10+)
SEARCH_WIDTH = 3                       # Beam width (1-20+)
OUTPUT_DIR = "occam_output"            # Directory for output files

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Data file: {DATA_FILE}")
print(f"  Search type: {SEARCH_TYPE}")
print(f"  Levels: {SEARCH_LEVELS}, Width: {SEARCH_WIDTH}")
print(f"  Output directory: {OUTPUT_DIR}")

Configuration:
  Data file: fire_data_split_all_signature_groups_nlcd_rebinned.txt
  Search type: full-up
  Levels: 7, Width: 3
  Output directory: occam_output


## 2. Load Data

Initialize the OCCAM manager and load your data:

In [6]:
# Initialize OCCAM manager
manager = pyoccam.VBMManager()

# Load data
if not manager.init_from_command_line(["occam", DATA_FILE]):
    raise RuntimeError(f"Could not load {DATA_FILE}")

# Display data information
print(f"✓ Loaded {DATA_FILE}")
print(f"  Sample size: {manager.get_sample_size()}")
print(f"  Has test data: {manager.has_test_data()}")

# Show variables
variables = manager.get_variable_list()
print(f"\nVariables ({len(variables)}):")
for i, var in enumerate(variables[:10], 1):  # Show first 10
    print(f"  {i:2d}. {var}")
if len(variables) > 10:
    print(f"  ... and {len(variables) - 10} more")

✓ Loaded fire_data_split_all_signature_groups_nlcd_rebinned.txt
  Sample size: 4937
  Has test data: False

Variables (23):
   1. in_d_l0
   2. mid_d_l0
   3. out_d_l0
   4. in_d_l1
   5. mid_d_l1
   6. out_d_l1
   7. in_d_l2
   8. mid_d_l2
   9. out_d_l2
  10. in_d_l3
  ... and 13 more


## 3. Configure Report Variables

Set which statistics to include in reports:

In [7]:
# Configure report columns
# Note: Don't include 'ID' or 'Model' - they're added automatically
report_vars = "Level$I, h, ddf, dLR, Alpha, %dH(DV), dAIC, dBIC"
manager.set_report_variables(report_vars)

print("Report configured with variables:")
print(f"  {report_vars}")

Report configured with variables:
  Level$I, h, ddf, dLR, Alpha, %dH(DV), dAIC, dBIC


## 4. Run Search

Search for the best models using your chosen algorithm:

In [8]:
# Run search
print(f"Running {SEARCH_TYPE} search...")
print(f"  Levels: {SEARCH_LEVELS}")
print(f"  Width: {SEARCH_WIDTH}")
print("-" * 60)

search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)

# Display search results (first 50 lines)
lines = search_report.split('\n')
for line in lines[:50]:
    print(line)
if len(lines) > 50:
    print(f"\n... ({len(lines) - 50} more lines)")

Running full-up search...
  Levels: 7
  Width: 3
------------------------------------------------------------
Searching levels:
1 : 22 new models, 3 kept; 4 total kept
2 : 60 new models, 3 kept; 7 total kept
3 : 61 new models, 3 kept; 10 total kept
4 : 64 new models, 3 kept; 13 total kept
5 : 69 new models, 3 kept; 16 total kept
6 : 79 new models, 3 kept; 19 total kept
7 : 91 new models, 3 kept; 22 total kept

  ID   MODEL                                      Level              H            dDF            dLR          Alpha        %dH(DV)           dAIC           dBIC
  22   IV:M0Z:I2Z:M2Z:O3Z:I4Z:O6Z:ElevZ               7         8.6541             47                        0.0000        19.1087       371.3430        65.6309
  21   IV:M0Z:I2Z:M2Z:I4Z:O4Z:O6Z:ElevZ               7         8.6544             47                        0.0000        19.0343       369.5308        63.8187
  20*  IV:M0Z:I2Z:M2Z:O3Z:I4Z:M6Z:ElevZ               7         8.6545             47                  

## 5. Identify Best Model

Extract the best model based on BIC:

In [ ]:
# Get best model
best_model = manager.get_best_model_by_bic()

if best_model:
    print(f"✨ Best model (by BIC): {best_model}")
    
    # Find its statistics in the search report
    for line in search_report.split('\n'):
        if best_model in line:
            print(f"\nModel statistics:")
            print(line)
            break
else:
    print("No best model found")
    best_model = "IV:CaseControl"

## 6. Fit Best Model

Generate detailed fit report for the best model:

In [ ]:
# Fit the best model
print(f"Fitting model: {best_model}")
print("-" * 60)

fit_report = manager.generate_fit_report(
    best_model,
    use_ipf_start=False,
    skip_trained_model_table=False,
    skip_ivi_tables=False
)

# Display key sections of fit report
fit_lines = fit_report.split('\n')
show_lines = False
line_count = 0
max_lines = 100

for line in fit_lines:
    # Show important sections
    if any(key in line for key in ["Model", "Percent", "Entropy", "Information", "Alpha"]):
        show_lines = True
    
    if show_lines and line_count < max_lines:
        print(line)
        line_count += 1
        
        # Stop after contingency table
        if "Degrees of" in line:
            show_lines = False

print(f"\n... (Full report has {len(fit_lines)} lines)")

## 7. Fit Custom Model (Optional)

You can also fit your own model specification:

In [ ]:
# Example: Fit a custom model
# Format: "IV:Var1Var2:Var3Var4" where variables form interaction terms

# Uncomment and modify this to fit your own model:
# custom_model = "IV:ApSxZ:EdZ:CZ"  # Example for dementia data
# custom_model = "IV:ABC:DEF"       # Generic example

# To fit a custom model, uncomment below:
"""
custom_model = "IV:YourModelHere"
print(f"Fitting custom model: {custom_model}")

try:
    custom_report = manager.generate_fit_report(
        custom_model,
        use_ipf_start=False,
        skip_trained_model_table=False,
        skip_ivi_tables=False
    )
    
    # Show summary
    for line in custom_report.split('\n')[:30]:
        print(line)
        
except Exception as e:
    print(f"Error fitting model: {e}")
"""

print("To fit a custom model, edit the cell above and uncomment the code.")

## 8. Save Results to Files

Save search and fit reports for later analysis:

In [ ]:
# Generate timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save search report
search_file = os.path.join(OUTPUT_DIR, f"search_{SEARCH_TYPE}_{timestamp}.txt")
with open(search_file, 'w') as f:
    f.write(search_report)
print(f"✓ Search report saved to: {search_file}")

# Save fit report
fit_file = os.path.join(OUTPUT_DIR, f"fit_{best_model.replace(':', '_')}_{timestamp}.txt")
with open(fit_file, 'w') as f:
    f.write(fit_report)
print(f"✓ Fit report saved to: {fit_file}")

print(f"\n📁 All results saved in: {os.path.abspath(OUTPUT_DIR)}")

## 9. Parse Results for Analysis

Extract search results into a pandas DataFrame for further analysis:

In [ ]:
# Parse search results into DataFrame
import pandas as pd

def parse_search_results(report):
    """Parse OCCAM search report into DataFrame"""
    lines = report.split('\n')
    data_lines = []
    
    # Find data section
    in_data = False
    for line in lines:
        if 'Model' in line and 'Level' in line:
            in_data = True
            continue
        if in_data and line.strip() and not line.startswith('#'):
            parts = line.split()
            if len(parts) >= 8:  # Valid data line
                data_lines.append(parts)
    
    if data_lines:
        # Create DataFrame
        columns = ['ID', 'Model', 'Level', 'h', 'ddf', 'dLR', 'Alpha', '%dH(DV)', 'dAIC', 'dBIC']
        df = pd.DataFrame(data_lines, columns=columns[:len(data_lines[0])])
        
        # Convert numeric columns
        for col in ['Level', 'ddf', 'dLR', 'Alpha', '%dH(DV)', 'dAIC', 'dBIC']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
    return None

# Parse and display results
df = parse_search_results(search_report)
if df is not None:
    print(f"Parsed {len(df)} models from search results\n")
    print("Top 10 models by dBIC:")
    print(df.nlargest(10, 'dBIC')[['Model', 'Level', 'dBIC', 'Alpha']])
else:
    print("Could not parse search results")

## 10. Summary

Analysis complete! You have:
- Searched for best models using OCCAM
- Identified the best model by BIC
- Generated detailed fit reports
- Saved results to files
- Parsed results for further analysis

### Next Steps:
1. Try different search algorithms (full-up, disjoint-up)
2. Adjust search parameters (levels, width)
3. Fit custom models based on domain knowledge
4. Visualize results (see visualization notebook)
5. Compare models across different datasets